# 🧬 Discovery Core: Colab Processing Pipeline

이 노트북은 로컬 리소스의 한계를 극복하기 위해 무거운 데이터 수집, 분자 전처리, 가상 스크리닝(도킹), AI 모델 추론 작업을 Google Colab GPU/고성능 CPU 환경에서 수행합니다.

**실행 순서**:
1. `[Run All]`을 눌러 전체 셀을 실행하세요.
2. 실행이 완료되면 맨 마지막 셀에서 `discovery_results.zip` 파일이 자동으로 다운로드됩니다.
3. 다운로드된 파일을 로컬 폴더에 넣고 `import_results.bat`를 실행하세요.

In [8]:
# @title 1. 환경 설정 및 코드 내려받기
import os

print("[INFO] 필수 시스템 패키지 설치 중...")
!apt-get update -qq
!apt-get install -y autodock-vina openbabel

print("[INFO] 저장소 클론 및 디렉토리 이동...")
if not os.path.exists('alphafold-drug-platform'):
    # 실제 저장소 URL로 변경 가능
    !git clone https://github.com/danjjak-ai/alphafold-drug-platform.git
os.chdir('alphafold-drug-platform')

print("[INFO] 파이썬 패키지 설치 중...")
!pip install -q -r requirements.txt
!pip install -q rdkit meeko deepchem torch torchvision torchaudio
print("환경 셋업 완료!")

[INFO] 필수 시스템 패키지 설치 중...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
autodock-vina is already the newest version (1.2.3-2).
openbabel is already the newest version (3.1.1+dfsg-6ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.
[INFO] 저장소 클론 및 디렉토리 이동...
Cloning into 'alphafold-drug-platform'...
remote: Enumerating objects: 1714, done.
remote: Counting objects: 100% (1714/1714), done.
remote: Compressing objects: 100% (1625/1625), done.
remote: Total 1714 (delta 104), reused 1678 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (1714/1714), 4.79 MiB | 4.08 MiB/s, done.
Resolving deltas: 100% (104/104), done.
[INFO] 파이썬 패키지 설치 중...
환경 셋업 완료!


In [ ]:
# @title 2. 파이프라인 대상 설정
DISEASE = "AChR-Associated Disorder" # @param {type:"string"}

print(f"설정된 타겟 질환: {DISEASE}")
os.environ['TARGET_DISEASE'] = DISEASE

설정된 타겟 질환: Cholinergic Crisis


In [10]:
# @title 3. 데이터 수집 단계 (DB Init, Target, Drug, Train Data)
print("[STEP 1] DB 초기화")
!python scripts/init_db.py

print("\n[STEP 2] 타겟 데이터 수집")
!python scripts/fetch_targets.py "$TARGET_DISEASE"

print("\n[STEP 3] 약물 데이터 수집")
!python scripts/fetch_drugs.py "$TARGET_DISEASE"

print("\n[STEP 4] 학습 데이터 수집")
!python scripts/fetch_training_data.py

[STEP 1] DB 초기화
Initializing database at: data/mg_discovery.db
Database initialization complete.

[STEP 2] 타겟 데이터 수집
Searching UniProt for targets related to: Cholinergic Crisis
Error searching UniProt: 400
No targets found for disease: Cholinergic Crisis

[STEP 3] 약물 데이터 수집
Searching ChEMBL for drugs related to targets: ['CHRNA1', 'MUSK', 'LRP4', 'HCRT', 'MOG', 'ZNF365'] or disease: Cholinergic Crisis
Fetching drugs for target gene: CHRNA1
Fetching drugs for target gene: MUSK
Fetching drugs for target gene: LRP4
Fetching drugs for target gene: HCRT
Fetching drugs for target gene: MOG
Fetching drugs for target gene: ZNF365
Searching ChEMBL by disease keyword: Cholinergic Crisis
Processing compounds: 100% 100/100 [00:00<00:00, 1055.25it/s]
Saved 86 compounds to database.

[STEP 4] 학습 데이터 수집
Fetching ChEMBL assay data for CHRNA1 (P02708)...
Found 343 activities for CHRNA1
Fetching ChEMBL assay data for MUSK (O15146)...
Found 27 activities for MUSK
Fetching ChEMBL assay data for LRP4 (O75

In [11]:
# @title 4. 구조 전처리 단계
print("[STEP 5] 타겟 단백질 구조 전처리")
!python scripts/prepare_targets.py

print("\n[STEP 6] 리간드 구조 전처리")
!python scripts/prepare_ligands.py

[STEP 5] 타겟 단백질 구조 전처리
Preparing CHRNA1 receptor...
  [구조 출처] RCSB 실험 구조 사용: data/structures/targets/7ql6_raw.pdb
Receptor data/structures/targets/chrna1.pdbqt Center: 143.62, 151.34, 142.64
Preparing MuSK receptor...
  [구조 출처] RCSB 실험 구조 사용: data/structures/targets/8s9p_raw.pdb
Receptor data/structures/targets/musk.pdbqt Center: 82.83, 81.01, 94.37
Preparing LRP4 receptor...
  [구조 출처] RCSB 실험 구조 사용: data/structures/targets/8s9p_raw.pdb
Receptor data/structures/targets/lrp4.pdbqt Center: 101.32, 99.30, 105.91

Target preparation complete.
TIP: AlphaFold DB 구조를 사용하려면 먼저 아래를 실행하세요:
  python scripts/predict_structures.py

[STEP 6] 리간드 구조 전처리
Traceback (most recent call last):
  File "/content/alphafold-drug-platform/alphafold-drug-platform/scripts/prepare_ligands.py", line 5, in <module>
    from meeko import MoleculePreparation
  File "/usr/local/lib/python3.12/dist-packages/meeko/__init__.py", line 27, in <module>
    from .polymer import Polymer
  File "/usr/local/lib/python3.12/dist-p

In [12]:
# @title 5. 가상 스크리닝 (AutoDock Vina)
print("[STEP 7] 도킹 시뮬레이션 (시간이 오래 걸릴 수 있습니다)")
!python scripts/run_docking.py

print("\n[STEP 8] 도킹 결과 분석")
!python scripts/post_docking_analysis.py

[STEP 7] 도킹 시뮬레이션 (시간이 오래 걸릴 수 있습니다)
Starting 33 docking tasks with 4 parallel workers (2 CPUs each)...
100% 33/33 [00:00<00:00, 4235.24it/s]

[STEP 8] 도킹 결과 분석
Connecting to database...
Found 143 hits with Vina score <= -7.0.
Calculating RDKit descriptor properties (ADMET proxy)...
Saved filtered candidates to results/top_candidates.csv
Saving to database table 'post_docking_filtered'...

Top 5 Candidates passing Lipinski:
       chembl_id gene_name  vina_score       QED
1     CHEMBL6293      LRP4      -9.941  0.443153
2    CHEMBL21333      MUSK      -9.629  0.431402
5     CHEMBL6293      MUSK      -9.081  0.443153
9   CHEMBL267864      LRP4      -8.925  0.739338
13   CHEMBL21333    CHRNA1      -8.747  0.431402


In [13]:
# @title 6. AI 모델 학습 및 추론
print("[STEP 9] 약물 활성 예측")
!python scripts/predict_activity.py

print("\n[STEP 10] 베이스라인 모델 학습")
!python scripts/train_baseline_model.py

print("\n[STEP 11] AI 딥러닝 모델 학습")
!python scripts/train_ai_model.py

[STEP 9] 약물 활성 예측
Loading Baseline AI Model...
Model not found! Run train_ai_model.py first.

[STEP 10] 베이스라인 모델 학습
Fetching all docking results for training...
Loaded 306 records. Applying Morgan Fingerprints...
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATION WARNING: please use MorganGenerator
[14:24:41] DEPRECATI

In [14]:
# @title 7. 결과 압축 및 로컬 다운로드
import os
from google.colab import files

print("결과 파일들을 압축합니다...")
!zip -r discovery_results.zip data/mg_discovery.db results/docking/ models/ -q

if os.path.exists('discovery_results.zip'):
    print("다운로드를 시작합니다. 브라우저에서 파일 저장을 허용해주세요.")
    files.download('discovery_results.zip')
else:
    print("압축 실패: 결과 파일이 생성되지 않았습니다.")

결과 파일들을 압축합니다...
다운로드를 시작합니다. 브라우저에서 파일 저장을 허용해주세요.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>